# 05_modeling

4주차 B팀 모델링 코드입니다. `modeling_dataset.csv`를 사용해 K-Means, 로지스틱 회귀, 랜덤 포레스트를 적용합니다.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

In [ ]:
df = pd.read_csv("../engineering/modeling_dataset.csv")
print(df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/modeling_dataset.csv'

## 1. 모델 입력 변수 설정

`jeonse_rate`와 `gap_rate`는 `gap_rate = 1 - jeonse_rate` 관계이므로, 중복을 줄이기 위해 모델 입력에서는 `gap_rate`를 사용합니다.

In [ ]:
features = [
    "gap_rate",
    "sale_growth_1m",
    "jeonse_growth_1m",
    "growth_gap_1m",
    "sale_volume_growth_1m",
    "rent_volume_growth_1m",
    "monthly_ratio"
]
target = "risk_target"
model_df = df.dropna(subset=features + [target]).copy()
X = model_df[features]
y = model_df[target].astype(int)
print(X.shape, y.shape)
print(y.value_counts())

## 2. 로지스틱 회귀와 랜덤 포레스트

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, random_state=42, class_weight="balanced", min_samples_leaf=2, n_jobs=-1
    )
}
records = []
for name, clf in models.items():
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    records.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0)
    })
    print("\n", name)
    print(confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred, zero_division=0))
model_comparison = pd.DataFrame(records)
model_comparison

## 3. 변수 중요도

In [ ]:
rf = models["Random Forest"]
feature_importance = pd.DataFrame({
    "feature": features,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)
feature_importance

In [ ]:
plt.figure(figsize=(9, 5))
fi = feature_importance.sort_values("importance")
plt.barh(fi["feature"], fi["importance"])
plt.xlabel("Feature Importance")
plt.title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()

## 4. K-Means 군집화

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
silhouette_scores = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    silhouette_scores.append({"k": k, "silhouette_score": silhouette_score(X_scaled, labels)})
pd.DataFrame(silhouette_scores)

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
model_df["cluster"] = kmeans.fit_predict(X_scaled)
summary_cols = features + ["user_risk_score", "investor_risk_score", "total_risk_score", "risk_target", "warning_flag"]
cluster_summary = model_df.groupby("cluster")[summary_cols].mean().reset_index()
cluster_summary["count"] = model_df.groupby("cluster").size().values
cluster_summary

In [ ]:
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)
model_df["pca_1"] = coords[:, 0]
model_df["pca_2"] = coords[:, 1]
plt.figure(figsize=(8, 6))
for c in sorted(model_df["cluster"].unique()):
    sub = model_df[model_df["cluster"] == c]
    plt.scatter(sub["pca_1"], sub["pca_2"], s=18, alpha=0.7, label=f"cluster {c}")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("K-Means Clusters (PCA 2D)")
plt.legend()
plt.tight_layout()
plt.show()

## 5. 결과 저장

In [ ]:
model_comparison.to_csv("../outputs/model_comparison.csv", index=False, encoding="utf-8-sig")
feature_importance.to_csv("../outputs/feature_importance.csv", index=False, encoding="utf-8-sig")
cluster_summary.to_csv("../outputs/kmeans_cluster_summary.csv", index=False, encoding="utf-8-sig")
print("저장 완료")